# Train Thermal UAV Detector (YOLOv12 + Ultralytics)

This notebook covers setup, optional Colab Drive mount, dataset verification, training, and validation.

In [2]:
from pathlib import Path
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('IN_COLAB =', IN_COLAB)

IN_COLAB = True


In [3]:
# Optional Colab setup
# from google.colab import drive
# drive.mount('/content/drive')

# Optional: clone your repo in Colab if needed
# !git clone <YOUR_REPO_URL> /content/VisionSentry

In [4]:
PROJECT_ROOT = Path('/content/VisionSentry') if IN_COLAB and Path('/content/VisionSentry').exists() else Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError(
        f'Cannot find project at {PROJECT_ROOT}. Set PROJECT_ROOT to your repo path.'
    )

print(f'Project root: {PROJECT_ROOT}')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')])

RuntimeError: Cannot find project at /content. Set PROJECT_ROOT to your repo path.

In [ ]:
import torch

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
cmd = [
    sys.executable, '-m', 'src.utils.dataset_checks',
    '--data', 'configs/dataset_thermal_uav.yaml'
]
subprocess.check_call(cmd, cwd=PROJECT_ROOT)

In [ ]:
train_cmd = [
    sys.executable, '-m', 'src.detection.train',
    '--config', 'configs/train_detector.yaml'
]
print('Running:', ' '.join(train_cmd))
subprocess.check_call(train_cmd, cwd=PROJECT_ROOT)

In [ ]:
detect_dir = PROJECT_ROOT / 'runs' / 'detect'
best_candidates = sorted(detect_dir.glob('**/weights/best.pt'))
if not best_candidates:
    raise FileNotFoundError('No best.pt found under runs/detect')
best_weights = best_candidates[-1]
print('Best weights:', best_weights)

In [ ]:
val_cmd = [
    sys.executable, '-m', 'src.detection.validate',
    '--weights', str(best_weights),
    '--data', 'configs/dataset_thermal_uav.yaml',
    '--split', 'val',
    '--imgsz', '960',
    '--batch', '16',
    '--device', '0',
    '--project', 'runs/val',
    '--name', 'thermal_uav_val_notebook'
]
print('Running:', ' '.join(val_cmd))
subprocess.check_call(val_cmd, cwd=PROJECT_ROOT)